# `manage_db` — TxGNN LaminDB instance setup

This notebook walks through every step required to initialise a fresh LaminDB
instance for the expanded TxGNN knowledge graph.

## Steps covered

| # | Task | Function |
|---|------|----------|
| 1 | Connect to the instance | `lamindb.connect()` |
| 2 | Pin bionty ontology source versions | `register_ontology_sources()` |
| 3 | Import bionty records into the instance | `bt.<Registry>.import_source()` |
| 4 | Register pertdb sources (laminlabs/pertdata) | `register_pertdb_sources()` |
| 5 | Query and transfer compounds from laminlabs/pertdata | `pt.Compound.connect()` |
| 6 | Inspect the custom `lnschema_txgnn` record types | — |
| 7 | Sync TxGNN nodes → LaminDB entities | `sync_txgnn_nodes_to_lamin_entities()` |

### Prerequisites

```bash
# Create and connect a LaminDB instance (run once)
lamin init --storage ./data/lamin_txgnn --modules bionty,pertdb,lnschema_txgnn
lamin connect myorg/myinstance
```

All functions live in `manage_db/` — import them directly.

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on the path so manage_db is importable.
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import lamindb as ln
import bionty as bt
import pertdb as pt

# manage_db helpers
from manage_db.register_ontology_sources import (
    register_ontology_sources,
    register_pertdb_sources,
    print_results,
)
from manage_db.sync_nodes_to_lamindb import sync_txgnn_nodes_to_lamin_entities

## 1. Connect to the instance

Pass the instance slug (`"account/name"`). Use `lamin connect` on the CLI
to set a default so you can omit the slug here.

In [ ]:
INSTANCE = "jkobject/txgnn_fresh2"   # ← change to your instance

ln.connect(INSTANCE)
print("Instance :", ln.setup.settings.instance.slug)
print("Storage  :", ln.setup.settings.instance.storage.root)

In [ ]:
# Quick audit: how many records exist before we do anything?
REGISTRIES = [
    ("bt.Gene",                      bt.Gene),
    ("bt.Disease",                   bt.Disease),
    ("bt.Tissue",                    bt.Tissue),
    ("bt.Phenotype",                 bt.Phenotype),
    ("bt.Pathway",                   bt.Pathway),
    ("bt.CellType",                  bt.CellType),
    ("bt.CellLine",                  bt.CellLine),
    ("bt.Organism",                  bt.Organism),
    ("pt.Compound",                  pt.Compound),
    ("pt.Biologic",                  pt.Biologic),
    ("pt.GeneticPerturbation",       pt.GeneticPerturbation),
    ("pt.EnvironmentalPerturbation", pt.EnvironmentalPerturbation),
]

pd.DataFrame(
    [(name, reg.objects.count()) for name, reg in REGISTRIES],
    columns=["registry", "records"],
)

## 2. Pin bionty ontology source versions

`register_ontology_sources()` does two things:

1. **Syncs** all public ontology releases from `bionty/base/sources.yaml` into
   the instance's `bt.Source` registry (without changing `currently_used` flags).
2. **Pins** each version listed in `_BIONTY_PINNED` by setting
   `source.currently_used = True` and saving — LaminDB automatically clears
   the flag on all other versions for the same registry.

### Pinned versions

| Registry | Source | Organism | Version |
|---|---|---|---|
| `bt.Gene` | ensembl | human | release-114 |
| `bt.Disease` | mondo | all | 2026-01-06 |
| `bt.Pathway` | go | all | 2025-10-10 |
| `bt.CellType` | cl | all | 2025-12-17 |
| `bt.Tissue` | uberon | all | 2025-12-04 |
| `bt.Phenotype` | hp | human | 2026-01-08 |
| `bt.CellLine` | cellosaurus | all | 53.0 |

> **Note on Phenotype**: we explicitly pin **HP** (Human Phenotype Ontology,
> ~20k terms), not PATO. All `effect/phenotype` nodes in the TxGNN KG use
> `node_source='HPO'` — PATO IDs would not match.

In [ ]:
# Dry-run first — see what would be pinned without writing anything.
dry_results = register_ontology_sources(dry_run=True)
print_results(dry_results, title="Bionty source pinning — DRY RUN")

In [ ]:
# Apply for real.
# On a fresh instance this takes ~10 s (syncs sources.yaml → DB).
results = register_ontology_sources(dry_run=False)
print_results(results, title="Bionty source pinning")

In [ ]:
# Inspect the Source registry to confirm what is currently_used.
bt.Source.filter(currently_used=True).to_dataframe()[
    ["entity", "name", "organism", "version", "currently_used"]
].sort_values("entity")

## 3. Import bionty records into the instance

`import_source()` downloads the pinned Parquet file and bulk-inserts all
records into the local database.  Call it once per registry.

> **Runtime**: ~5–10 minutes total for all registries on a laptop.

In [ ]:
TO_IMPORT = [
    # (registry,         organism)  — use organism only where required
    (bt.Gene,            "human"),   # Ensembl human release-114 → 91k genes
    (bt.Disease,         None),      # MONDO → 30k diseases
    (bt.Pathway,         None),      # GO   → 48k pathways
    (bt.CellType,        None),      # CL   → 3k cell types
    (bt.Tissue,          None),      # UBERON → 15k tissues
    (bt.Phenotype,       "human"),   # HP (human) → 20k phenotypes
    (bt.CellLine,        None),      # Cellosaurus → 167k cell lines
]

for registry, organism in TO_IMPORT:
    entity_name = f"bionty.{registry.__name__}"
    source = bt.Source.filter(entity=entity_name, currently_used=True).one_or_none()
    if source is None:
        print(f"  ⚠ {entity_name}: no currently_used source — run step 2 first")
        continue
    before = registry.objects.count()
    kwargs = {"source": source, "ignore_conflicts": True}
    if organism:
        kwargs["organism"] = organism
    registry.import_source(**kwargs)
    after = registry.objects.count()
    print(f"  ✓  {entity_name:<35}  {before:>8,} → {after:>8,}  (+{after - before:,})")

## 4. Register pertdb sources

`register_pertdb_sources()` records a `bt.Source` entry for each pertdb
registry pointing to the `laminlabs/pertdata` LaminDB instance — the
authoritative, curated database for chemical and biological perturbations.

Unlike bionty ontologies, pertdb does not have a versioned `sources.yaml`.
The *version* we record is the **date of the snapshot** we pull from
`laminlabs/pertdata` (step 5).  Update it whenever you re-sync.

### Registered registries

| Registry | Description |
|---|---|
| `pt.Compound` | Chemical compounds / drugs (ChEMBL, DrugBank) |
| `pt.Biologic` | Biologic perturbations (antibodies, cytokines, …) |
| `pt.GeneticPerturbation` | Genetic perturbations (CRISPR, RNAi, ORF, …) |
| `pt.EnvironmentalPerturbation` | Environmental perturbations (media, conditions) |
| `pt.CompoundPerturbation` | Compound perturbation events (dose, context) |
| `pt.CombinationPerturbation` | Combination perturbation events |

In [ ]:
# Dry-run to preview.
pt_dry = register_pertdb_sources(version="2026-03-02", dry_run=True)
print_results(pt_dry, title="pertdb source registration — DRY RUN")

In [ ]:
# Register for real.  Pass the version date so it matches the snapshot you pull in step 5.
PERTDATA_VERSION = "2026-03-02"   # update when re-syncing

pt_results = register_pertdb_sources(version=PERTDATA_VERSION, dry_run=False)
print_results(pt_results, title="pertdb source registration")

In [ ]:
# Confirm in the Source registry.
bt.Source.filter(entity__startswith="pertdb").to_dataframe()[
    ["entity", "name", "version", "organism", "currently_used", "source_website"]
]

## 5. Query and transfer compounds from `laminlabs/pertdata`

The `laminlabs/pertdata` instance is publicly readable.  Use
`pt.Compound.connect("laminlabs/pertdata")` for live cross-instance queries,
or bulk-fetch and save locally for offline use.

### 5a — Live cross-instance query (read-only, no local changes)

In [ ]:
# Query compounds from laminlabs/pertdata without touching the local instance.
remote_compounds = pt.Compound.connect("laminlabs/pertdata")

print("Remote compound count  :", remote_compounds.count())

# Peek at the first 5.
sample = remote_compounds[:5].to_dataframe()[
    ["uid", "name", "ontology_id", "chembl_id", "type"]
]
sample

In [ ]:
# Search for a specific compound by name (fuzzy search).
pt.Compound.connect("laminlabs/pertdata").search("imatinib").to_dataframe()[
    ["name", "ontology_id", "chembl_id", "smiles"]
].head()

### 5b — Bulk-transfer to local instance

For offline use or when building the TxGNN KG, copy all records locally.
We fetch as dicts from the remote, reconnect locally, then bulk-create.

In [ ]:
COMPOUND_FIELDS = (
    "uid", "name", "ontology_id", "abbr", "synonyms",
    "description", "type", "chembl_id", "smiles",
    "canonical_smiles", "inchikey", "molweight", "molformula", "moa",
)

# ── Fetch from remote (stays connected to laminlabs/pertdata) ──────────────
remote_qs  = pt.Compound.connect("laminlabs/pertdata")
total      = remote_qs.count()
print(f"Fetching {total:,} compounds from laminlabs/pertdata …")

CHUNK = 2_000
remote_dicts = []
for start in range(0, total, CHUNK):
    remote_dicts.extend(remote_qs[start : start + CHUNK].values(*COMPOUND_FIELDS))
print(f"Fetched {len(remote_dicts):,} records")

# ── Re-connect to local instance and bulk-create ───────────────────────────
ln.connect(INSTANCE)

existing_uids = set(pt.Compound.objects.values_list("uid", flat=True))
print(f"Already in local instance: {len(existing_uids):,}")

buffer: list[pt.Compound] = []
created = skipped = 0
SAVE_CHUNK = 500

for d in remote_dicts:
    if d["uid"] in existing_uids:
        skipped += 1
        continue
    kwargs = {k: d[k] for k in COMPOUND_FIELDS if d.get(k) is not None}
    buffer.append(pt.Compound(**kwargs))
    if len(buffer) >= SAVE_CHUNK:
        pt.Compound.objects.bulk_create(buffer, ignore_conflicts=True)
        created += len(buffer)
        buffer = []

if buffer:
    pt.Compound.objects.bulk_create(buffer, ignore_conflicts=True)
    created += len(buffer)

print(f"Created : {created:,}")
print(f"Skipped : {skipped:,}")
print(f"Total   : {pt.Compound.objects.count():,}")

## 6. Custom `lnschema_txgnn` record types

Five node types in the TxGNN KG are not covered by bionty or pertdb.  They
are registered as custom `SQLRecord` subclasses in `manage_db/lnschema_txgnn/`:

| Model | Node type | Primary ID |
|---|---|---|
| `Paper` | `paper` | PubMed ID / DOI |
| `Transcript` | `transcript` | Ensembl ENST… |
| `Enhancer` | `enhancer` | ENCODE ID |
| `Dataset` | `dataset` | Internal UUID / DOI |
| `Mutation` | `mutation` | dbSNP rsID / HGVS |

> CellLine is already covered by `bionty.CellLine` (Cellosaurus).

In [ ]:
# The module must be installed as a schema module in the LaminDB instance.
# If lamin init was called with --modules lnschema_txgnn, the tables already exist.

# Import and inspect the models.
import sys
sys.path.insert(0, str(ROOT / "manage_db"))
import lnschema_txgnn as txgnn_schema

for model in [txgnn_schema.Paper, txgnn_schema.Transcript,
              txgnn_schema.Enhancer, txgnn_schema.Dataset, txgnn_schema.Mutation]:
    fields = [f.name for f in model._meta.get_fields() if hasattr(f, 'column')]
    print(f"  {model.__name__:<15}  fields: {', '.join(fields)}")

In [ ]:
# Example: create a Paper record.
paper = txgnn_schema.Paper(
    pmid="37778123",
    doi="10.1038/s41591-023-02558-7",
    title="Prediction of drug-disease associations using graph neural networks",
    year=2023,
    journal="Nature Medicine",
).save()

print("Saved Paper uid:", paper.uid)
print(txgnn_schema.Paper.filter(pmid="37778123").one())

In [ ]:
# Record counts in each custom registry.
pd.DataFrame([
    {"model": m.__name__, "records": m.objects.count()}
    for m in [txgnn_schema.Paper, txgnn_schema.Transcript,
               txgnn_schema.Enhancer, txgnn_schema.Dataset, txgnn_schema.Mutation]
])

## 7. Sync TxGNN nodes → LaminDB entities

`sync_txgnn_nodes_to_lamin_entities()` reads the legacy `nodes.tab` file,
maps each node to its canonical bionty or pertdb registry, looks up
existing records, and creates missing ones.

### Input format (`nodes.tab`)

| Column | Example |
|---|---|
| `node_index` | 0 |
| `node_id` | 51177 |
| `node_type` | gene/protein |
| `node_name` | ANAMORSIN |
| `node_source` | NCBI |

### Node type mapping

| Legacy type | Registry |
|---|---|
| `gene/protein` | `bionty.Gene` (NCBI gene ID) |
| `drug` | `pertdb.Compound` (DrugBank ID) |
| `disease` | `bionty.Disease` (MONDO ID) |
| `effect/phenotype` | `bionty.Phenotype` (HP ID) |
| `anatomy` | `bionty.Tissue` (UBERON ID) |
| `pathway` | `bionty.Pathway` (GO / Reactome ID) |
| `exposure` | `pertdb.EnvironmentalPerturbation` |
| `biological_process` | `bionty.Pathway` (GO ID) |
| `molecular_function` | `bionty.Pathway` (GO ID) |
| `cellular_component` | `bionty.Pathway` (GO ID) |

In [ ]:
# Dry-run — inspect the mapping without touching the database.
mapping_dry = sync_txgnn_nodes_to_lamin_entities(
    nodes_path=ROOT / "data" / "txdata" / "nodes.tab",
    mapping_output_path=None,   # skip writing a CSV
    dry_run=True,
)

print("Shape:", mapping_dry.shape)
print()
print("Status breakdown:")
print(mapping_dry["status"].value_counts().to_string())
print()
print("Registry breakdown:")
print(mapping_dry["registry"].value_counts().to_string())

In [ ]:
# Peek at uncertain nodes (no ontology ID → cannot reliably match).
uncertain = mapping_dry[mapping_dry["status"] == "uncertain"]
print(f"Uncertain nodes: {len(uncertain):,}")
uncertain[["node_type", "node_name", "node_source", "registry", "key_field"]].head(10)

In [ ]:
# Run the real sync (writes records to the database).
# This may take 5–15 minutes depending on instance size.
mapping = sync_txgnn_nodes_to_lamin_entities(
    nodes_path=ROOT / "data" / "txdata" / "nodes.tab",
    mapping_output_path=ROOT / "data" / "txdata" / "node_entity_mapping.csv",
    dry_run=False,
)

print("Status breakdown:")
print(mapping["status"].value_counts().to_string())

In [ ]:
# Per-node-type coverage table.
summary = (
    mapping
    .groupby(["node_type", "status"])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda df: df.sum(axis=1))
    .sort_values("total", ascending=False)
)
summary

In [ ]:
# Spot-check: verify a mapped disease node resolves in the LaminDB registry.
sample_disease = mapping[
    (mapping["node_type"] == "disease") & (mapping["status"] == "existing")
].iloc[0]

print("Node      :", sample_disease[["node_name", "node_id", "registry", "key_value"]])

disease_record = bt.Disease.filter(
    ontology_id=sample_disease["key_value"]
).one_or_none()
print("LaminDB   :", disease_record)

## Final record counts

After all steps above, the instance should contain:

In [ ]:
ln.connect(INSTANCE)

final = [
    ("bt.Gene",                      bt.Gene),
    ("bt.Disease",                   bt.Disease),
    ("bt.Tissue",                    bt.Tissue),
    ("bt.Phenotype",                 bt.Phenotype),
    ("bt.Pathway",                   bt.Pathway),
    ("bt.CellType",                  bt.CellType),
    ("bt.CellLine",                  bt.CellLine),
    ("bt.Organism",                  bt.Organism),
    ("pt.Compound",                  pt.Compound),
    ("pt.Biologic",                  pt.Biologic),
    ("pt.GeneticPerturbation",       pt.GeneticPerturbation),
    ("pt.EnvironmentalPerturbation", pt.EnvironmentalPerturbation),
]

pd.DataFrame(
    [(name, reg.objects.count()) for name, reg in final],
    columns=["registry", "records"],
)